In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, TrainingArguments
from sklearn.metrics.pairwise import cosine_similarity
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments, Trainer, default_data_collator
)
from datasets import Dataset
import sys
import sentencepiece
import tiktoken
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
import re
import os
import pickle
import joblib
import os
from google.colab import drive

In [2]:
df = pd.read_csv("cleaned_data.csv")
df

,QuestionText,Category,Answer
0,"['أيهما', 'أفضل', 'الدراسة', 'السابق', 'الوقت']",التعليم,"['الدراسة', 'الوقت', 'الحالي', 'تعتبر', 'أفضل'..."
1,"['أليس', 'القطن', 'عماد', 'الثروة']",الاقتصاد والعمل,"['القطن', 'يعتبر', 'أهم', 'المنتجات', 'الزراعي..."
2,"['أتصعد', 'الشمس']",التعليم,"['الشمس', 'تصعد', 'الشرق']"
3,"['أتعرف', 'البكتيريا', 'بأنها', 'كائنات', 'حية']",التعليم,"['البكتيريا', 'تعرف', 'بأنها', 'كائنات', 'حية'..."
4,"['أيتكون', 'الهواء', 'أساسا']",التعليم,"['الهواء', 'يتكون', 'أساسا', 'النيتروجين']"
...,...,...,...
5004,"['يحدث', 'تسخين']",العلوم,"['تسخين', 'تتسارع', 'حركة', 'جزيئاته', 'يؤدي',..."
5005,"['الهدف', 'اتفاقية', 'التجارة', 'الحرة', 'لأمر...",الاقتصاد والعمل,"['الهدف', 'اتفاقية', 'التجارة', 'الحرة', 'لأمر..."
5006,"['الغرض', 'اتفاقية', 'التجارة', 'الحرة', 'لأمر...",الاقتصاد والعمل,"['الغرض', 'اتفاقية', 'التجارة', 'الحرة', 'لأمر..."
5007,"['هدف', 'إعجاز', 'القرآن', 'وفقا', 'للمعتقد']",الدين,"['يؤدي', 'الإعجاز', 'غرضين', 'رئيسين', 'الأول'..."


In [3]:
df['input_text'] = ( df['QuestionText'] .str.replace("[", "", regex=False) .str.replace("]", "", regex=False) .str.replace("'", "", regex=False) .str.replace(",", " ", regex=False) )
df['target_text'] = ( df['Answer'] .str.replace("[", "", regex=False) .str.replace("]", "", regex=False) .str.replace("'", "", regex=False) .str.replace(",", " ", regex=False) )
df.head()

,QuestionText,Category,Answer,input_text,target_text
0,"['أيهما', 'أفضل', 'الدراسة', 'السابق', 'الوقت']",التعليم,"['الدراسة', 'الوقت', 'الحالي', 'تعتبر', 'أفضل'...",أيهما أفضل الدراسة السابق الوقت,الدراسة الوقت الحالي تعتبر أفضل بسبب توف...
1,"['أليس', 'القطن', 'عماد', 'الثروة']",الاقتصاد والعمل,"['القطن', 'يعتبر', 'أهم', 'المنتجات', 'الزراعي...",أليس القطن عماد الثروة,القطن يعتبر أهم المنتجات الزراعية ويعد ا...
2,"['أتصعد', 'الشمس']",التعليم,"['الشمس', 'تصعد', 'الشرق']",أتصعد الشمس,الشمس تصعد الشرق
3,"['أتعرف', 'البكتيريا', 'بأنها', 'كائنات', 'حية']",التعليم,"['البكتيريا', 'تعرف', 'بأنها', 'كائنات', 'حية'...",أتعرف البكتيريا بأنها كائنات حية,البكتيريا تعرف بأنها كائنات حية دقيقة
4,"['أيتكون', 'الهواء', 'أساسا']",التعليم,"['الهواء', 'يتكون', 'أساسا', 'النيتروجين']",أيتكون الهواء أساسا,الهواء يتكون أساسا النيتروجين


**TF-IDF EMBEDDING**

In [ ]:
# Add start and end tokens for decoder
df['target_text'] = df['answer_text'].apply(
    lambda x: 'start ' + x + ' end'
)


X_train, X_test, y_train, y_test = train_test_split(
    df['input_text'],
    df['target_text'],
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

df[['input_text', 'answer_text', 'target_text']].head()

Training data: (4007,)
Testing data: (1002,)


,input_text,answer_text,target_text
0,أيهما أفضل الدراسة السابق الوقت,الدراسة الوقت الحالي تعتبر أفضل بسبب توف...,start الدراسة الوقت الحالي تعتبر أفضل بسب...
1,أليس القطن عماد الثروة,القطن يعتبر أهم المنتجات الزراعية ويعد ا...,start القطن يعتبر أهم المنتجات الزراعية و...
2,أتصعد الشمس,الشمس تصعد الشرق,start الشمس تصعد الشرق end
3,أتعرف البكتيريا بأنها كائنات حية,البكتيريا تعرف بأنها كائنات حية دقيقة,start البكتيريا تعرف بأنها كائنات حية دقي...
4,أيتكون الهواء أساسا,الهواء يتكون أساسا النيتروجين,start الهواء يتكون أساسا النيتروجين end


In [ ]:
# TF-IDF vectorizer for questions
question_vectorizer = TfidfVectorizer(
    token_pattern=r"(?u)\b\w+\b"
)

X_train_tfidf = question_vectorizer.fit_transform(X_train)
X_test_tfidf = question_vectorizer.transform(X_test)

tfidf_features = X_train_tfidf.shape[1]

print("TF-IDF train shape:", X_train_tfidf.shape)
print("TF-IDF test shape:", X_test_tfidf.shape)
print("TF-IDF features:", tfidf_features)

TF-IDF train shape: (4007, 4454)
TF-IDF test shape: (1002, 4454)
TF-IDF features: 4454


In [ ]:
# Initialize TF-IDF vectorizer for Answer
answer_vectorizer = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b")

# Fit and transform on Answer
answer_tfidf_matrix = answer_vectorizer.fit_transform(df['target_text'])

# Convert to DataFrame for readability
answer_tfidf = pd.DataFrame(
    answer_tfidf_matrix.toarray(),
    columns=answer_vectorizer.get_feature_names_out()
)

# Show TF-IDF values for answers
answer_tfidf

,0,1,10,100,1000,1048,11,1127م,1171م,1174م,...,يومي,يوميا,يومية,يومين,يونس,يونيو,يوهان,يوهانس,ييلينا,ﷺ
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5004,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
5005,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
5006,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
5007,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.199129


Tokenization and padding

In [ ]:
# Use the Tokenizer() to convert the words into numerical representation

input_tokenizer = Tokenizer() #Build vocabulary and convert words into numbers (just in the vocab until now)
input_tokenizer.fit_on_texts(X_train)

target_tokenizer = Tokenizer() #Build vocabulary and convert words into numbers (just in the vocab until now)
target_tokenizer.fit_on_texts(y_train)


# Convert each sentence into a sequence of integers
X_train_seq = input_tokenizer.texts_to_sequences(X_train)
X_test_seq = input_tokenizer.texts_to_sequences(X_test)

Y_train_seq = target_tokenizer.texts_to_sequences(y_train)
Y_test_seq = target_tokenizer.texts_to_sequences(y_test)


# Ensure uniform input size for batch processing - Padding using pad_sequences
max_encoder_len = max(len(seq) for seq in X_train_seq) #To make all questions the same length
max_decoder_len = max(len(seq) for seq in Y_train_seq) #To make all answers the same length

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_encoder_len,
    padding='post'
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_encoder_len,
    padding='post'
)

Y_train_pad = pad_sequences(
    Y_train_seq,
    maxlen=max_decoder_len,
    padding='post'
)

Y_test_pad = pad_sequences(
    Y_test_seq,
    maxlen=max_decoder_len,
    padding='post'
)

print("Max encoder length:", max_encoder_len)
print("Max decoder length:", max_decoder_len)
print("X_train_pad shape:", X_train_pad.shape)
print("Y_train_pad shape:", Y_train_pad.shape)

Max encoder length: 18
Max decoder length: 102
X_train_pad shape: (4007, 18)
Y_train_pad shape: (4007, 102)


Preparing decoder input and target

In [ ]:
# Create input-output pairs where the model learns to predict the next word

decoder_input_data = Y_train_pad[:, :-1] #All answer tokens except the last one
decoder_target_data = Y_train_pad[:, 1:] #All answer tokens except the first one


# Reshape for sparse loss
decoder_target_data = decoder_target_data.reshape(
    decoder_target_data.shape[0],
    decoder_target_data.shape[1],
    1
)


# Get the vocabulary size for the encoder and decoder
encoder_vocab_size = len(input_tokenizer.word_index) + 1
decoder_vocab_size = len(target_tokenizer.word_index) + 1


# Display the sizes
print("Encoder vocab:", encoder_vocab_size)
print("Decoder vocab:", decoder_vocab_size)
print("Encoder shape:", X_train_pad.shape)
print("Decoder input shape:", decoder_input_data.shape)
print("Decoder target shape:", decoder_target_data.shape)

Encoder vocab: 4455
Decoder vocab: 11705
Encoder shape: (4007, 18)
Decoder input shape: (4007, 101)
Decoder target shape: (4007, 101, 1)


build encoder

In [ ]:
# Define an Input layer that has the shape of the max length for the sentence

encoder_inputs = Input(shape=(max_encoder_len,))


# Define an Embedding layer

enc_emb = Embedding( # embedding for each word
    input_dim=encoder_vocab_size,
    output_dim=128,
    mask_zero=True
)(encoder_inputs) 


# Use SimpleRNN as hidden layer

encoder_rnn = SimpleRNN(
    256,
    return_state=True
)


# Produces a final state / context vector

encoder_outputs, encoder_state = encoder_rnn(enc_emb) #question vector

encoder_outputs, encoder_state

(<KerasTensor shape=(None, 256), dtype=float32, sparse=False, ragged=False, name=keras_tensor_40>,
 <KerasTensor shape=(None, 256), dtype=float32, sparse=False, ragged=False, name=keras_tensor_41>)

build decoder

In [ ]:
# Define an Input layer for the decoder

decoder_inputs = Input(shape=(max_decoder_len - 1,))


# Define an Embedding layer for decoder vocabulary

dec_emb_layer = Embedding(
    input_dim=decoder_vocab_size,
    output_dim=128,
    mask_zero=True
)


# Apply embedding

dec_emb = dec_emb_layer(decoder_inputs)


# Use SimpleRNN as decoder hidden layer

decoder_rnn = SimpleRNN(
    256,
    return_sequences=True,
    return_state=True
)


# Pass encoder state to decoder

decoder_outputs, _ = decoder_rnn(
    dec_emb,
    initial_state=encoder_state
)


# Dense layer to predict the next word

decoder_dense = Dense(
    decoder_vocab_size,
    activation='softmax'
)


decoder_outputs = decoder_dense(decoder_outputs)

**RNN**

In [ ]:
# Build the model

model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)


# Compile the model

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


# Show the model summary

model.summary()


# Train the model

model.fit(
    [X_train_pad, decoder_input_data],
    decoder_target_data,
    batch_size=64,
    epochs=10,
    validation_split=0.2
)

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_9       │ (None, 18)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_10      │ (None, 101)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_4         │ (None, 18, 128)   │    570,240 │ input_layer_9[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_6         │ (None, 18)        │          0 │ input_layer_9[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_5         │ (None, 101, 128)  │  1,498,240 │ input_layer_10[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn_4        │ [(None, 256),     │     98,560 │ embedding_4[0][0… │
│ (SimpleRNN)         │ (None, 256)]      │            │ not_equal_6[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn_5        │ [(None, 101,      │     98,560 │ embedding_5[0][0… │
│ (SimpleRNN)         │ 256), (None,      │            │ simple_rnn_4[0][… │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 101,       │  3,008,185 │ simple_rnn_5[0][… │
│                     │ 11705)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,273,785 (20.12 MB)

 Trainable params: 5,273,785 (20.12 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 12s 131ms/step - accuracy: 0.0992 - loss: 8.4098 - val_accuracy: 0.0708 - val_loss: 8.0471
Epoch 2/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 3s 59ms/step - accuracy: 0.1267 - loss: 7.7849 - val_accuracy: 0.1404 - val_loss: 7.7815
Epoch 3/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 3s 58ms/step - accuracy: 0.1410 - loss: 7.3657 - val_accuracy: 0.1406 - val_loss: 7.7908
Epoch 4/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 3s 58ms/step - accuracy: 0.1426 - loss: 7.2148 - val_accuracy: 0.1419 - val_loss: 7.7084
Epoch 5/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 3s 58ms/step - accuracy: 0.1429 - loss: 7.0752 - val_accuracy: 0.1431 - val_loss: 7.5985
Epoch 6/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 3s 60ms/step - accuracy: 0.1447 - loss: 6.9195 - val_accuracy: 0.1432 - val_loss: 7.6580
Epoch 7/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 3s 59ms/step - accuracy: 0.1475 - loss: 6.7716 - val_accuracy: 0.1470 - val_loss: 7.5213
Epoch 8/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 3s 58ms/step - accuracy: 0.1523 - loss: 6.5456 - val_accuracy: 0.1501 -

Encoder and decoder for prediction

In [ ]:
# Encoder inference model

encoder_model = Model(
    encoder_inputs,
    encoder_state
)


# Decoder inference model

decoder_state_input = Input(shape=(256,))

decoder_single_input = Input(shape=(1,))

dec_emb2 = dec_emb_layer(decoder_single_input)

decoder_outputs2, state_h2 = decoder_rnn(
    dec_emb2,
    initial_state=decoder_state_input
)

decoder_outputs2 = decoder_dense(decoder_outputs2)

decoder_model = Model(
    [decoder_single_input, decoder_state_input],
    [decoder_outputs2, state_h2]
)

In [ ]:
def decode_sequence(input_seq):

    state = encoder_model.predict(input_seq, verbose=0)

    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = target_tokenizer.word_index['start']

    decoded = ""

    max_len = 50

    for _ in range(max_len):

        output_tokens, state = decoder_model.predict(
            [target_seq, state],
            verbose=0
        )

        probs = output_tokens[0, -1, :]

        sampled_index = np.argmax(probs)

        word = target_tokenizer.index_word.get(sampled_index, '')

        if word == 'end' or word == '':
            break

        decoded += " " + word

        target_seq[0, 0] = sampled_index

    return decoded.strip()

Test it

In [ ]:
for i in range(5):

    print("\n====================")

    print("QUESTION:")
    print(X_test.iloc[i])

    input_seq = X_test_pad[i].reshape(1, max_encoder_len)

    print("\nREAL ANSWER:")
    print(y_test.iloc[i])

    print("\nPREDICTED ANSWER:")
    print(decode_sequence(input_seq))


QUESTION:
علام  تعتمد  تحقيق

REAL ANSWER:
start أعتمد  تحقيق  أهدافي  التخطيط  الجيد  والعمل  الجاد  والاستمرار  التعلم  والتطوير  الشخصي end

PREDICTED ANSWER:
الإجابة تعتمد الشخص

QUESTION:
أهم  الدول  المستوردة  للمنتجات  الزراعية

REAL ANSWER:
start يعتبر  القطاع  الزراعي  القطاعات  المهمة  وتلعب  الزراعة  دورا  هاما  المنظومة  الاقتصادية  والاجتماعية  للمجتمعات  الريفية  ترتبط  ارتباطا  وثيقا  بجهود  المحافظة  البيئة  الطبيعية  واستمراريتها  يواجه  القطاع  الزراعي  الأردن  مشاكل  وتحديات  متمثلة  توالي  سنوات  تذبذب  قلة  الأراضي  ندرة  الموارد  والمخاطر  المختلفة  يساهم  القطاع  الزراعي  نسبته  الناتج  المحلي  الإجمالي  ويعمل  مجموع  القوى  وتشكل  الصادرات  الزراعية  مجموع  صادرات  يذهب  الأسواق  العربية  حقق  الأردن  الاكتفاء  الذاتي  عدد  المنتجات  الزراعية  كزيت  الكثير  المنتجات  الغذائية  الأساسية  كأنواع  القمح  ومشتقات  الحليب  والسكر  اللحوم  الحمراء  والخضراوات  تستورد  الخارج  نمت  الصادرات  المنتجات  الزراعية  بنسبة  عام  2005  مقارنة  قيمته  عام  2004  أهم  الصادرات

In [ ]:
def clean_answer_for_bleu(text):
    text = str(text)
    text = re.sub(r'\bstart\b', '', text)
    text = re.sub(r'\bend\b', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# ==============================
# BLEU Score for RNN Seq2Seq
# ==============================

sample_size = 100

n = min(sample_size, len(X_test))

rnn_predictions = []
real_answers = []
questions = []

for i in range(n):

    input_seq = X_test_pad[i].reshape(1, max_encoder_len)

    predicted_answer = decode_sequence(input_seq)

    real_answer = clean_answer_for_bleu(y_test.iloc[i])

    rnn_predictions.append(predicted_answer)
    real_answers.append(real_answer)
    questions.append(X_test.iloc[i])


references = [
    [answer.split()]
    for answer in real_answers
]

hypotheses = [
    prediction.split()
    for prediction in rnn_predictions
]

smoothing = SmoothingFunction().method1

rnn_bleu_score = corpus_bleu(
    references,
    hypotheses,
    smoothing_function=smoothing
)

print("RNN Seq2Seq BLEU Score:", rnn_bleu_score)

RNN Seq2Seq BLEU Score: 0.001972426606374887


**BERT EMBEDDING**

PREPARE INPUT AND TARGET

In [ ]:
df['input_text'] = (
    df['QuestionText']
    .astype(str)
    .str.replace("[", "", regex=False)
    .str.replace("]", "", regex=False)
    .str.replace("'", "", regex=False)
    .str.replace(",", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df['answer_text'] = (
    df['Answer']
    .astype(str)
    .str.replace("[", "", regex=False)
    .str.replace("]", "", regex=False)
    .str.replace("'", "", regex=False)
    .str.replace(",", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# use safe special tokens
df['target_text'] = df['answer_text'].apply(
    lambda x: 'start ' + x + ' end'
)


# Splitting the data into training - testing
X_train, X_test, y_train, y_test = train_test_split(
    df['input_text'],
    df['target_text'],
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

df[['input_text', 'answer_text', 'target_text']].head()

Training data: (4007,)
Testing data: (1002,)


,input_text,answer_text,target_text
0,أيهما أفضل الدراسة السابق الوقت,الدراسة الوقت الحالي تعتبر أفضل بسبب توفر التك...,start الدراسة الوقت الحالي تعتبر أفضل بسبب توف...
1,أليس القطن عماد الثروة,القطن يعتبر أهم المنتجات الزراعية ويعد الأعمدة...,start القطن يعتبر أهم المنتجات الزراعية ويعد ا...
2,أتصعد الشمس,الشمس تصعد الشرق,start الشمس تصعد الشرق end
3,أتعرف البكتيريا بأنها كائنات حية,البكتيريا تعرف بأنها كائنات حية دقيقة,start البكتيريا تعرف بأنها كائنات حية دقيقة end
4,أيتكون الهواء أساسا,الهواء يتكون أساسا النيتروجين,start الهواء يتكون أساسا النيتروجين end


In [ ]:
# Load AraBERT tokenizer and model

bert_model_name = "aubmindlab/bert-base-arabertv02"

bert_tokenizer = AutoTokenizer.from_pretrained(bert_model_name)

bert_model = AutoModel.from_pretrained(bert_model_name)


# Move model to GPU if available

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

bert_model = bert_model.to(device)

bert_model.eval()

print("Device:", device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Device: cuda


In [ ]:
# Load AraBERT tokenizer and model

bert_model_name = "aubmindlab/bert-base-arabertv02"

bert_tokenizer = AutoTokenizer.from_pretrained(bert_model_name)

bert_model = AutoModel.from_pretrained(bert_model_name)


# Move model to GPU if available

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

bert_model = bert_model.to(device)

bert_model.eval()

print("Device:", device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Device: cuda


In [ ]:
def get_bert_embeddings(texts, batch_size=16):

    all_embeddings = []

    texts = list(texts)

    for start in range(0, len(texts), batch_size):

        batch_texts = texts[start:start + batch_size]

        inputs = bert_tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        inputs = {key: value.to(device) for key, value in inputs.items()}

        with torch.no_grad():

            outputs = bert_model(**inputs)

        token_embeddings = outputs.last_hidden_state

        attention_mask = inputs["attention_mask"]

        mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

        sentence_embeddings = torch.sum(token_embeddings * mask, dim=1) / torch.clamp(
            mask.sum(dim=1),
            min=1e-9
        )

        all_embeddings.append(sentence_embeddings.cpu().numpy())

    all_embeddings = np.vstack(all_embeddings)

    return all_embeddings

In [ ]:
X_train_bert = get_bert_embeddings(X_train, batch_size=16)

X_test_bert = get_bert_embeddings(X_test, batch_size=16)


# Get BERT embedding dimension

bert_dim = X_train_bert.shape[1]


print("X_train_bert shape:", X_train_bert.shape)
print("X_test_bert shape:", X_test_bert.shape)
print("BERT dimension:", bert_dim)

X_train_bert shape: (4007, 768)
X_test_bert shape: (1002, 768)
BERT dimension: 768


Reshape for RNN

In [ ]:
X_train_bert_rnn = X_train_bert.reshape(
    X_train_bert.shape[0],
    1,
    bert_dim
)

X_test_bert_rnn = X_test_bert.reshape(
    X_test_bert.shape[0],
    1,
    bert_dim
)


print("X_train_bert_rnn shape:", X_train_bert_rnn.shape)
print("X_test_bert_rnn shape:", X_test_bert_rnn.shape)

X_train_bert_rnn shape: (4007, 1, 768)
X_test_bert_rnn shape: (1002, 1, 768)


TOKENIZATION

In [ ]:
target_tokenizer = Tokenizer()

target_tokenizer.fit_on_texts(y_train)


Y_train_seq = target_tokenizer.texts_to_sequences(y_train)
Y_test_seq = target_tokenizer.texts_to_sequences(y_test)

print("start index:", target_tokenizer.word_index.get('start'))
print("end index:", target_tokenizer.word_index.get('end'))

start index: 1
end index: 2


PADDING

In [ ]:

max_decoder_len = 50

Y_train_pad = pad_sequences(
    Y_train_seq,
    maxlen=max_decoder_len,
    padding='post',
    truncating='post'
)

Y_test_pad = pad_sequences(
    Y_test_seq,
    maxlen=max_decoder_len,
    padding='post',
    truncating='post'
)

print("Max decoder length:", max_decoder_len)
print("Y_train_pad shape:", Y_train_pad.shape)
print("Y_test_pad shape:", Y_test_pad.shape)

Max decoder length: 50
Y_train_pad shape: (4007, 50)
Y_test_pad shape: (1002, 50)


Prepare encoder and decoder

In [ ]:
# Create input-output pairs where the model learns to predict the next word

decoder_input_data = Y_train_pad[:, :-1]

decoder_target_data = Y_train_pad[:, 1:]


# Reshape for sparse loss

decoder_target_data = decoder_target_data.reshape(
    decoder_target_data.shape[0],
    decoder_target_data.shape[1],
    1
)


# Get the vocabulary size for the decoder

decoder_vocab_size = len(target_tokenizer.word_index) + 1


# Display the sizes

print("Decoder vocab:", decoder_vocab_size)
print("Encoder shape:", X_train_bert_rnn.shape)
print("Decoder input shape:", decoder_input_data.shape)
print("Decoder target shape:", decoder_target_data.shape)

Decoder vocab: 11705
Encoder shape: (4007, 1, 768)
Decoder input shape: (4007, 49)
Decoder target shape: (4007, 49, 1)


Encoder

In [ ]:
encoder_inputs = Input(shape=(1, bert_dim))


encoder_rnn = SimpleRNN(
    256,
    return_state=True
)


encoder_outputs, encoder_state = encoder_rnn(encoder_inputs)

encoder_outputs, encoder_state

(<KerasTensor shape=(None, 256), dtype=float32, sparse=False, ragged=False, name=keras_tensor_56>,
 <KerasTensor shape=(None, 256), dtype=float32, sparse=False, ragged=False, name=keras_tensor_57>)

Decoder

In [ ]:
decoder_inputs = Input(shape=(max_decoder_len - 1,))

dec_emb_layer = Embedding(
    input_dim=decoder_vocab_size,
    output_dim=128,
    mask_zero=True
)


# Apply embedding

dec_emb = dec_emb_layer(decoder_inputs)


# Use SimpleRNN as decoder hidden layer

decoder_rnn = SimpleRNN(
    256,
    return_sequences=True,
    return_state=True
)


# Pass encoder state to decoder

decoder_outputs, state_h = decoder_rnn(
    dec_emb,
    initial_state=encoder_state
)


# Dense layer to predict the next word

decoder_dense = Dense(decoder_vocab_size,activation='softmax')


decoder_outputs = decoder_dense(decoder_outputs)

In [ ]:
model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)


model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


model.summary()


# Train the model

history = model.fit(
    [X_train_bert_rnn, decoder_input_data],
    decoder_target_data,
    batch_size=32,
    epochs=5,
    validation_split=0.2
)

Model: "functional_9"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_14      │ (None, 49)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_13      │ (None, 1, 768)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_6         │ (None, 49, 128)   │  1,498,240 │ input_layer_14[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn_6        │ [(None, 256),     │    262,400 │ input_layer_13[0… │
│ (SimpleRNN)         │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn_7        │ [(None, 49, 256), │     98,560 │ embedding_6[0][0… │
│ (SimpleRNN)         │ (None, 256)]      │            │ simple_rnn_6[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 49, 11705) │  3,008,185 │ simple_rnn_7[0][… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,867,385 (18.57 MB)

 Trainable params: 4,867,385 (18.57 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
101/101 ━━━━━━━━━━━━━━━━━━━━ 10s 54ms/step - accuracy: 0.1115 - loss: 8.0272 - val_accuracy: 0.1439 - val_loss: 7.6591
Epoch 2/5
101/101 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.1450 - loss: 7.1918 - val_accuracy: 0.1456 - val_loss: 7.5242
Epoch 3/5
101/101 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.1514 - loss: 6.8225 - val_accuracy: 0.1563 - val_loss: 7.4474
Epoch 4/5
101/101 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.1625 - loss: 6.4282 - val_accuracy: 0.1707 - val_loss: 7.3160
Epoch 5/5
101/101 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.1811 - loss: 6.0099 - val_accuracy: 0.1834 - val_loss: 7.2334


Encoder inference

In [ ]:
encoder_model = Model(
    encoder_inputs,
    encoder_state
)

Decoder inference

In [ ]:
decoder_state_input = Input(shape=(256,))
decoder_single_input = Input(shape=(1,))
dec_emb2 = dec_emb_layer(decoder_single_input)


decoder_outputs2, state_h2 = decoder_rnn(
    dec_emb2,
    initial_state=decoder_state_input
)

decoder_outputs2 = decoder_dense(decoder_outputs2)

decoder_model = Model(
    [decoder_single_input, decoder_state_input],
    [decoder_outputs2, state_h2]
)

In [ ]:
# DECODING FUNCTION

def decode_sequence(input_seq):

    state = encoder_model.predict(input_seq, verbose=0)

    target_seq = np.zeros((1, 1))

    target_seq[0, 0] = target_tokenizer.word_index['start']

    decoded = ""

    max_len = 50

    for _ in range(max_len):

        output_tokens, state = decoder_model.predict(
            [target_seq, state],
            verbose=0
        )

        probs = output_tokens[0, -1, :]

        sampled_index = np.argmax(probs)

        word = target_tokenizer.index_word.get(sampled_index, '')

        if word == 'end' or word == '':
            break

        decoded += " " + word

        target_seq[0, 0] = sampled_index

    return decoded.strip()

In [ ]:
# Training

for i in range(5):

    print("\n====================")

    print("QUESTION:")
    print(X_test.iloc[i])

    input_seq = X_test_bert_rnn[i].reshape(1, 1, bert_dim)

    print("\nREAL ANSWER:")
    print(y_test.iloc[i])

    print("\nPREDICTED ANSWER:")
    print(decode_sequence(input_seq))


QUESTION:
علام تعتمد تحقيق

REAL ANSWER:
start أعتمد تحقيق أهدافي التخطيط الجيد والعمل الجاد والاستمرار التعلم والتطوير الشخصي end

PREDICTED ANSWER:
أعتمد تحسين الشخص

QUESTION:
أهم الدول المستوردة للمنتجات الزراعية

REAL ANSWER:
start يعتبر القطاع الزراعي القطاعات المهمة وتلعب الزراعة دورا هاما المنظومة الاقتصادية والاجتماعية للمجتمعات الريفية ترتبط ارتباطا وثيقا بجهود المحافظة البيئة الطبيعية واستمراريتها يواجه القطاع الزراعي الأردن مشاكل وتحديات متمثلة توالي سنوات تذبذب قلة الأراضي ندرة الموارد والمخاطر المختلفة يساهم القطاع الزراعي نسبته الناتج المحلي الإجمالي ويعمل مجموع القوى وتشكل الصادرات الزراعية مجموع صادرات يذهب الأسواق العربية حقق الأردن الاكتفاء الذاتي عدد المنتجات الزراعية كزيت الكثير المنتجات الغذائية الأساسية كأنواع القمح ومشتقات الحليب والسكر اللحوم الحمراء والخضراوات تستورد الخارج نمت الصادرات المنتجات الزراعية بنسبة عام 2005 مقارنة قيمته عام 2004 أهم الصادرات الزراعية الزيوت والسجائر أهم الدول المستوردة الإمارات العربية وسوريا end

PREDICTED ANSWER:
تعد اضطراب الإي

In [ ]:
def clean_answer_for_bleu(text):

    text = str(text)

    text = re.sub(r'\bstart\b', '', text)

    text = re.sub(r'\bend\b', '', text)

    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
# Initialize an empty list to store the model predictions

bert_rnn_predictions = []

actual_answers = []

sample_size = 100

n = min(sample_size, len(X_test))


# Loop to get the predictions for each sentence in the testing data

for i in range(n):

    input_seq = X_test_bert_rnn[i].reshape(1, 1, bert_dim)

    prediction = decode_sequence(input_seq)

    bert_rnn_predictions.append(prediction)

    actual_answers.append(clean_answer_for_bleu(y_test.iloc[i]))


# Format the output into tokenized list for both actual and predictions

references = [
    [answer.split()]
    for answer in actual_answers
]

hypotheses = [
    prediction.split()
    for prediction in bert_rnn_predictions
]


# Initialize smoothing function

smoothing = SmoothingFunction().method1


# Calculate BLEU score

bert_rnn_bleu_score = corpus_bleu(
    references,
    hypotheses,
    smoothing_function=smoothing
)


print("BERT + RNN Seq2Seq BLEU Score:", bert_rnn_bleu_score)

BERT + RNN Seq2Seq BLEU Score: 0.006240430604810626


In [ ]:
# Folder in Google Drive
export_dir = "/content/drive/MyDrive/bert_rnn_seq2seq_model"

os.makedirs(export_dir, exist_ok=True)


# Save the main trained model
model.save(f"{export_dir}/model.keras")


# Save encoder and decoder inference models
encoder_model.save(f"{export_dir}/encoder_model.keras")

decoder_model.save(f"{export_dir}/decoder_model.keras")


# Save target tokenizer
with open(f"{export_dir}/target_tokenizer.pkl", "wb") as file:
    pickle.dump(target_tokenizer, file)


# Save BERT tokenizer and BERT model
bert_tokenizer.save_pretrained(f"{export_dir}/bert_tokenizer")

bert_model.save_pretrained(f"{export_dir}/bert_embedding_model")


# Save BERT embeddings
np.save(f"{export_dir}/X_train_bert.npy", X_train_bert)

np.save(f"{export_dir}/X_test_bert.npy", X_test_bert)


# Save important values
model_info = {
    "bert_model_name": bert_model_name,
    "bert_dim": bert_dim,
    "max_decoder_len": max_decoder_len,
    "decoder_vocab_size": decoder_vocab_size
}

joblib.dump(
    model_info,
    f"{export_dir}/model_info.pkl"
)

print("Saved folder:", export_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved folder: /content/drive/MyDrive/bert_rnn_seq2seq_model


**Transformer QA**

In [4]:
# Prepare the input questions
df['input_text'] = (
    df['QuestionText']
    .str.replace("[", "", regex=False)
    .str.replace("]", "", regex=False)
    .str.replace("'", "", regex=False)
    .str.replace(",", " ", regex=False)
)

# Prepare the target answers
df['target_text'] = (
    df['Answer']
    .str.replace("[", "", regex=False)
    .str.replace("]", "", regex=False)
    .str.replace("'", "", regex=False)
    .str.replace(",", " ", regex=False)
)

df.head()

,QuestionText,Category,Answer,input_text,target_text
0,"['أيهما', 'أفضل', 'الدراسة', 'السابق', 'الوقت']",التعليم,"['الدراسة', 'الوقت', 'الحالي', 'تعتبر', 'أفضل'...",أيهما أفضل الدراسة السابق الوقت,الدراسة الوقت الحالي تعتبر أفضل بسبب توف...
1,"['أليس', 'القطن', 'عماد', 'الثروة']",الاقتصاد والعمل,"['القطن', 'يعتبر', 'أهم', 'المنتجات', 'الزراعي...",أليس القطن عماد الثروة,القطن يعتبر أهم المنتجات الزراعية ويعد ا...
2,"['أتصعد', 'الشمس']",التعليم,"['الشمس', 'تصعد', 'الشرق']",أتصعد الشمس,الشمس تصعد الشرق
3,"['أتعرف', 'البكتيريا', 'بأنها', 'كائنات', 'حية']",التعليم,"['البكتيريا', 'تعرف', 'بأنها', 'كائنات', 'حية'...",أتعرف البكتيريا بأنها كائنات حية,البكتيريا تعرف بأنها كائنات حية دقيقة
4,"['أيتكون', 'الهواء', 'أساسا']",التعليم,"['الهواء', 'يتكون', 'أساسا', 'النيتروجين']",أيتكون الهواء أساسا,الهواء يتكون أساسا النيتروجين


In [5]:
X = df['input_text']
y = df['target_text']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)

y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (4007,)
Testing data: (1002,)


**T5 model**

In [ ]:
model_name = "google/mt5-small"

t5_tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=False
)

t5_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [6]:
def calculate_bleu_score(generate_function, X_test, y_test, model_name, sample_size=None):

    X_list = X_test.tolist() if hasattr(X_test, "tolist") else list(X_test)
    y_list = y_test.tolist() if hasattr(y_test, "tolist") else list(y_test)

    if sample_size is not None:
        X_list = X_list[:sample_size]
        y_list = y_list[:sample_size]

    predictions = []

    for question in X_list:
        predicted_answer = generate_function(question)
        predictions.append(predicted_answer)

    references = [
        [str(actual_answer).split()]
        for actual_answer in y_list
    ]

    hypotheses = [
        str(predicted_answer).split()
        for predicted_answer in predictions
    ]

    smoothing = SmoothingFunction().method1

    bleu_score = corpus_bleu(
        references,
        hypotheses,
        smoothing_function=smoothing
    )

    predictions_df = pd.DataFrame({
        "Model": model_name,
        "Question": X_list,
        "Actual_Answer": y_list,
        "Predicted_Answer": predictions
    })

    return bleu_score, predictions_df

In [ ]:
def tokenize_function(questions, answers):

    questions = ["question: " + str(q) for q in questions]
    answers = [str(a) for a in answers]

    model_inputs = t5_tokenizer(
        questions,
        padding=True,
        truncation=True,
        max_length=128
    )

    labels = t5_tokenizer(
        answers,
        padding=True,
        truncation=True,
        max_length=128
    )

    labels_ids = labels["input_ids"]

    # Replace padding token ids in labels with -100
    labels_ids = [
        [
            token if token != t5_tokenizer.pad_token_id else -100
            for token in label
        ]
        for label in labels_ids
    ]

    model_inputs["labels"] = labels_ids

    return model_inputs

In [ ]:
## Tokenize training data:
train_encodings = tokenize_function(X_train, y_train)

## Tokenize testing data:
test_encodings = tokenize_function(X_test, y_test)

In [ ]:
## Convert tokenized data into Hugging Face Dataset format:
train_dataset = Dataset.from_dict({
    "input_ids": train_encodings["input_ids"],
    "attention_mask": train_encodings["attention_mask"],
    "labels": train_encodings["labels"]
})

test_dataset = Dataset.from_dict({
    "input_ids": test_encodings["input_ids"],
    "attention_mask": test_encodings["attention_mask"],
    "labels": test_encodings["labels"]
})

In [ ]:
## Define training configuration:
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5_qa_results",

    learning_rate=2e-5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    num_train_epochs=3,

    weight_decay=0.01,

    predict_with_generate=True,

    logging_steps=50,

    save_strategy="epoch",

    report_to="none"
)

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=t5_tokenizer,
    model=t5_model
)

trainer = Seq2SeqTrainer(
    model=t5_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator
)

In [ ]:
## Start training:
trainer.train()

Step,Training Loss
50,24.601523
100,20.818621
150,18.431697
200,15.667772
250,14.143372
300,12.703807
350,11.544834
400,10.384746
450,9.686553
500,9.126266


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3006, training_loss=6.972005965308356, metrics={'train_runtime': 1269.1101, 'train_samples_per_second': 9.472, 'train_steps_per_second': 2.369, 'total_flos': 434499025766400.0, 'train_loss': 6.972005965308356, 'epoch': 3.0})

In [ ]:
def generate_t5_answer(question):

    input_text = "question: " + str(question)

    inputs = t5_tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {key: value.to(t5_model.device) for key, value in inputs.items()}

    t5_model.eval()

    with torch.no_grad():
        outputs = t5_model.generate(
            **inputs,
            max_length=128,
            num_beams=4
        )

    answer = t5_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer.strip()

In [ ]:
## Test T5 model using 5 samples from the test set:

for i in range(5):

    print("\n====================")

    print("QUESTION:")
    print(X_test.iloc[i])

    print("\nREAL ANSWER:")
    print(y_test.iloc[i])

    print("\nT5 PREDICTED ANSWER:")
    print(generate_t5_answer(X_test.iloc[i]))


QUESTION:
علام  تعتمد  تحقيق

REAL ANSWER:
أعتمد  تحقيق  أهدافي  التخطيط  الجيد  والعمل  الجاد  والاستمرار  التعلم  والتطوير  الشخصي

T5 PREDICTED ANSWER:
تعتمد تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق ت

QUESTION:
أهم  الدول  المستوردة  للمنتجات  الزراعية

REAL ANSWER:
يعتبر  القطاع  الزراعي  القطاعات  المهمة  وتلعب  الزراعة  دورا  هاما  المنظومة  الاقتصادية  والاجتماعية  للمجتمعات  الريفية  ترتبط  ارتباطا  وثيقا  بجهود  المحافظة  البيئة  الطبيعية  واستمراريتها  يواجه  القطاع  الزراعي  الأردن  مشاكل  وتحديات  متمثلة  توالي  سنوات  تذبذب  قلة  الأراضي  ندرة  الموارد  والمخاطر  المختلفة  يساهم  القطاع  الزراعي  نسبته  الناتج  المحلي  الإجمالي  ويعمل  مجموع  الق

In [ ]:
## Test T5 model using a custom question:

custom_question = "ما هي أهمية التعليم؟"

print("QUESTION:")
print(custom_question)

print("\nT5 PREDICTED ANSWER:")
print(generate_t5_answer(custom_question))

QUESTION:
ما هي أهمية التعليم؟

T5 PREDICTED ANSWER:
<extra_id_0> التعليم التعليم


In [ ]:
t5_bleu_score, t5_predictions_df = calculate_bleu_score(
    generate_function=generate_t5_answer,
    X_test=X_test,
    y_test=y_test,
    model_name="mT5-small",
    sample_size=100
)

print("mT5 BLEU Score:", t5_bleu_score)

t5_predictions_df.to_csv("t5_predictions.csv", index=False)

t5_predictions_df.head()

mT5 BLEU Score: 0.06816589070822245


,Model,Question,Actual_Answer,Predicted_Answer
0,mT5-small,علام تعتمد تحقيق,أعتمد تحقيق أهدافي التخطيط الجيد والعمل ...,تعتمد تحقيق تحقيق تحقيق تحقيق تحقيق تحقيق تحقي...
1,mT5-small,أهم الدول المستوردة للمنتجات الزراعية,يعتبر القطاع الزراعي القطاعات المهمة وتلع...,<extra_id_0> الدول المستوردة للمنتجات الزراعية
2,mT5-small,أهمية التعليم,أهمية التعليم العالي تشمل تطوير المهارات ...,<extra_id_0> التعليم التعليم
3,mT5-small,لماذا الأكسجين,الأكسجين مهم لأنه ضروري لعملية التنفس ال...,يعتمد الأكسجين الأكسجين
4,mT5-small,مادة تعتبر موصل جيد,النحاس يعتبر موصل جيد للكهرباء,تعتمد مادة تعتبر موصل جيد تعتبر موصل جيد


In [ ]:
save_path = "/content/drive/MyDrive/mt5_qa_model"

trainer.save_model(save_path)
t5_tokenizer.save_pretrained(save_path)

print("mT5 model saved in:", save_path)
print("Current folder:", os.getcwd())

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

mT5 model saved in: /content/drive/MyDrive/mt5_qa_model
Current folder: /content


**QWEN model**

In [7]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

qwen_tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

qwen_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float32,
    trust_remote_code=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
qwen_model = qwen_model.to(device)

if qwen_tokenizer.pad_token is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

qwen_model.config.pad_token_id = qwen_tokenizer.pad_token_id

print("Qwen model loaded successfully on:", device)
print("Pad token id:", qwen_tokenizer.pad_token_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen model loaded successfully on: cuda
Pad token id: 151643


In [ ]:
def tokenize_qwen_function(questions, answers):

    input_ids_list = []
    attention_mask_list = []
    labels_list = []

    for question, answer in zip(questions, answers):

        prompt = "السؤال: " + str(question) + "\nالإجابة:"

        full_text = prompt + " " + str(answer) + qwen_tokenizer.eos_token

        tokenized_full = qwen_tokenizer(
            full_text,
            padding="max_length",
            truncation=True,
            max_length=256
        )

        tokenized_prompt = qwen_tokenizer(
            prompt,
            truncation=True,
            max_length=256
        )

        input_ids = tokenized_full["input_ids"]
        attention_mask = tokenized_full["attention_mask"]

        labels = input_ids.copy()

        prompt_length = len(tokenized_prompt["input_ids"])

        # Ignore question/prompt tokens during loss calculation
        labels[:prompt_length] = [-100] * prompt_length

        # Ignore padding tokens
        labels = [
            token if mask == 1 else -100
            for token, mask in zip(labels, attention_mask)
        ]

        input_ids_list.append(input_ids)
        attention_mask_list.append(attention_mask)
        labels_list.append(labels)

    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_mask_list,
        "labels": labels_list
    }

In [9]:
train_encodings = tokenize_qwen_function(
    X_train.tolist(),
    y_train.tolist()
)

test_encodings = tokenize_qwen_function(
    X_test.tolist(),
    y_test.tolist()
)

In [10]:
train_dataset = Dataset.from_dict(train_encodings)

test_dataset = Dataset.from_dict(test_encodings)

In [11]:
training_args = TrainingArguments(
    output_dir="./qwen_qa_results",

    learning_rate=2e-5,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    num_train_epochs=5,

    logging_steps=25,

    save_strategy="epoch",
    eval_strategy="no",

    fp16=torch.cuda.is_available(),

    report_to="none"
)

In [ ]:
def generate_qwen_answer(question):

    prompt = "السؤال: " + str(question) + "\nالإجابة:"

    inputs = qwen_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    inputs = {key: value.to(qwen_model.device) for key, value in inputs.items()}

    qwen_model.eval()

    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=qwen_tokenizer.eos_token_id
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]

    answer = qwen_tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )

    return answer.strip()

In [13]:
trainer = Trainer(
    model=qwen_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=default_data_collator
)


In [14]:
trainer.train()

Step,Training Loss
25,2.549429
50,2.112794
75,2.060359
100,2.050405
125,2.312747
150,2.226584
175,2.212200
200,2.175511
225,2.252359
250,2.018334


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=20035, training_loss=0.6650933160518868, metrics={'train_runtime': 6708.6806, 'train_samples_per_second': 2.986, 'train_steps_per_second': 2.986, 'total_flos': 2.202774120628224e+16, 'train_loss': 0.6650933160518868, 'epoch': 5.0})

In [15]:
qwen_bleu_score, qwen_predictions_df = calculate_bleu_score(
    generate_function=generate_qwen_answer,
    X_test=X_test,
    y_test=y_test,
    model_name="Qwen2.5-0.5B-Instruct",
    sample_size=100
)

print("Qwen BLEU Score:", qwen_bleu_score)

qwen_predictions_df.to_csv("qwen_predictions.csv", index=False)

qwen_predictions_df.head()

Qwen BLEU Score: 0.20851312478631157


,Model,Question,Actual_Answer,Predicted_Answer
0,Qwen2.5-0.5B-Instruct,علام تعتمد تحقيق,أعتمد تحقيق أهدافي التخطيط الجيد والعمل ...,أعتمد تحقيق أهدافي التخطيط الجيد والعمل ...
1,Qwen2.5-0.5B-Instruct,أهم الدول المستوردة للمنتجات الزراعية,يعتبر القطاع الزراعي القطاعات المهمة وتلع...,أهم الدول المستوردة للمنتجات الزراعية الع...
2,Qwen2.5-0.5B-Instruct,أهمية التعليم,أهمية التعليم العالي تشمل تطوير المهارات ...,أهمية التعليم العالي تشمل تطوير المهارات ...
3,Qwen2.5-0.5B-Instruct,لماذا الأكسجين,الأكسجين مهم لأنه ضروري لعملية التنفس ال...,الأكسجين ضروري لأنه يعزز الصحة يزيد الطا...
4,Qwen2.5-0.5B-Instruct,مادة تعتبر موصل جيد,النحاس يعتبر موصل جيد للكهرباء,الرمل يعتبر موصل جيد وناعم


Testing on 5 samples

In [16]:
## Test the model using 5 samples:
for i in range(5):

    print("\n====================")

    print("QUESTION:")
    print(X_test.iloc[i])

    print("\nREAL ANSWER:")
    print(y_test.iloc[i])

    print("\nQWEN PREDICTED ANSWER:")
    print(generate_qwen_answer(X_test.iloc[i]))


QUESTION:
علام  تعتمد  تحقيق

REAL ANSWER:
أعتمد  تحقيق  أهدافي  التخطيط  الجيد  والعمل  الجاد  والاستمرار  التعلم  والتطوير  الشخصي

QWEN PREDICTED ANSWER:
أعتمد  تحقيق  أهدافي  التخطيط  الجيد  والعمل  الجاد  والاستمرار  التعلم  والتطوير  الشخصي

QUESTION:
أهم  الدول  المستوردة  للمنتجات  الزراعية

REAL ANSWER:
يعتبر  القطاع  الزراعي  القطاعات  المهمة  وتلعب  الزراعة  دورا  هاما  المنظومة  الاقتصادية  والاجتماعية  للمجتمعات  الريفية  ترتبط  ارتباطا  وثيقا  بجهود  المحافظة  البيئة  الطبيعية  واستمراريتها  يواجه  القطاع  الزراعي  الأردن  مشاكل  وتحديات  متمثلة  توالي  سنوات  تذبذب  قلة  الأراضي  ندرة  الموارد  والمخاطر  المختلفة  يساهم  القطاع  الزراعي  نسبته  الناتج  المحلي  الإجمالي  ويعمل  مجموع  القوى  وتشكل  الصادرات  الزراعية  مجموع  صادرات  يذهب  الأسواق  العربية  حقق  الأردن  الاكتفاء  الذاتي  عدد  المنتجات  الزراعية  كزيت  الكثير  المنتجات  الغذائية  الأساسية  كأنواع  القمح  ومشتقات  الحليب  والسكر  اللحوم  الحمراء  والخضراوات  تستورد  الخارج  نمت  الصادرات  المنتجات  الزراعية

Testing on a custome question

In [17]:
question = "ما هي أهمية التعليم؟"

print("QUESTION:")
print(question)

print("\nQWEN ANSWER:")
print(generate_qwen_answer(question))

QUESTION:
ما هي أهمية التعليم؟

QWEN ANSWER:
أهمية  التعليم  العالي  تشمل  تطوير  المهارات  تعزيز  القدرة  التفكير  توفير  فرص  عمل  ودعم  البحث  العلمي  والابتكار


In [ ]:
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [19]:
import os

print(os.listdir("/content/drive/MyDrive"))

['2024-11-22 20-50-05.mp4', '2024-12-01 20-31-55.mp4', 'New Microsoft Word Document.docx', 'Colab Notebooks']


In [20]:
save_path = "/content/drive/MyDrive/Colab Notebooks/qwen_qa_model"

os.makedirs(save_path, exist_ok=True)

qwen_model.save_pretrained(save_path)
qwen_tokenizer.save_pretrained(save_path)

print("Saved in:", save_path)
print(os.listdir(save_path))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved in: /content/drive/MyDrive/Colab Notebooks/qwen_qa_model
['config.json', 'generation_config.json', 'model.safetensors', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json']


In [21]:
!du -sh "/content/drive/MyDrive/Colab Notebooks/qwen_qa_model"

1.9G	/content/drive/MyDrive/Colab Notebooks/qwen_qa_model


**GPT model**

In [ ]:
model_name = "aubmindlab/aragpt2-base"

gpt_tokenizer = AutoTokenizer.from_pretrained(model_name)

if gpt_tokenizer.pad_token is None:
    gpt_tokenizer.pad_token = gpt_tokenizer.eos_token

gpt_model = AutoModelForCausalLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpt_model = gpt_model.to(device)

gpt_model.config.pad_token_id = gpt_tokenizer.pad_token_id
gpt_model.config.use_cache = False

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.50M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.52M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/553M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: aubmindlab/aragpt2-base
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def tokenize_gpt_function(questions, answers):

    input_ids_list = []
    attention_mask_list = []
    labels_list = []

    for question, answer in zip(questions, answers):

        prompt = "السؤال: " + str(question) + "\nالإجابة:"

        full_text = prompt + " " + str(answer) + gpt_tokenizer.eos_token

        tokenized_full = gpt_tokenizer(
            full_text,
            padding="max_length",
            truncation=True,
            max_length=256
        )

        tokenized_prompt = gpt_tokenizer(
            prompt,
            truncation=True,
            max_length=256
        )

        input_ids = tokenized_full["input_ids"]
        attention_mask = tokenized_full["attention_mask"]

        labels = input_ids.copy()

        prompt_length = len(tokenized_prompt["input_ids"])

        # Ignore question/prompt tokens during loss calculation
        labels[:prompt_length] = [-100] * prompt_length

        # Ignore padding tokens
        labels = [
            token if mask == 1 else -100
            for token, mask in zip(labels, attention_mask)
        ]

        input_ids_list.append(input_ids)
        attention_mask_list.append(attention_mask)
        labels_list.append(labels)

    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_mask_list,
        "labels": labels_list
    }

In [ ]:
train_encodings = tokenize_gpt_function(
    X_train.tolist(),
    y_train.tolist()
)

test_encodings = tokenize_gpt_function(
    X_test.tolist(),
    y_test.tolist()
)

In [ ]:
train_dataset = Dataset.from_dict(train_encodings)

test_dataset = Dataset.from_dict(test_encodings)

In [ ]:
training_args = TrainingArguments(
    output_dir="./aragpt2_qa_results",

    learning_rate=2e-5,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    num_train_epochs=1,

    weight_decay=0.01,

    logging_steps=50,

    save_strategy="epoch",

    report_to="none",

    fp16=False
)

In [ ]:
trainer = Trainer(
    model=gpt_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=default_data_collator
)

trainer.train()

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
50,6.897390
100,4.627829
150,4.404529
200,4.194247
250,3.803491
300,3.811518
350,3.525835
400,3.603579
450,3.381457
500,3.238324


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4007, training_loss=2.851124790621006, metrics={'train_runtime': 414.8501, 'train_samples_per_second': 9.659, 'train_steps_per_second': 9.659, 'total_flos': 523498586112000.0, 'train_loss': 2.851124790621006, 'epoch': 1.0})

In [ ]:
save_path = "/content/drive/MyDrive/aragpt2_qa_model"

trainer.save_model(save_path)
gpt_tokenizer.save_pretrained(save_path)

print("Current folder:", os.getcwd())

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Current folder: /content


In [ ]:
def generate_gpt_answer(question):

    prompt = "السؤال: " + str(question) + "\nالإجابة:"

    inputs = gpt_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    inputs = {key: value.to(gpt_model.device) for key, value in inputs.items()}

    gpt_model.eval()

    with torch.no_grad():
        outputs = gpt_model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=gpt_tokenizer.eos_token_id
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]

    answer = gpt_tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )

    return answer.strip()

In [ ]:
## Test the model using 5 samples:
for i in range(5):

    print("\n====================")

    print("QUESTION:")
    print(X_test.iloc[i])

    print("\nREAL ANSWER:")
    print(y_test.iloc[i])

    print("\nQWEN PREDICTED ANSWER:")
    print(generate_gpt_answer(X_test.iloc[i]))


QUESTION:
علام  تعتمد  تحقيق

REAL ANSWER:
أعتمد  تحقيق  أهدافي  التخطيط  الجيد  والعمل  الجاد  والاستمرار  التعلم  والتطوير  الشخصي

QWEN PREDICTED ANSWER:
أعتمد  تحقيق  الأهداف  المهنية  خلال  ممارسة  الأنشطة  المهنية  وتقديم  الدعم  المستمر  للموظفين

QUESTION:
أهم  الدول  المستوردة  للمنتجات  الزراعية

REAL ANSWER:
يعتبر  القطاع  الزراعي  القطاعات  المهمة  وتلعب  الزراعة  دورا  هاما  المنظومة  الاقتصادية  والاجتماعية  للمجتمعات  الريفية  ترتبط  ارتباطا  وثيقا  بجهود  المحافظة  البيئة  الطبيعية  واستمراريتها  يواجه  القطاع  الزراعي  الأردن  مشاكل  وتحديات  متمثلة  توالي  سنوات  تذبذب  قلة  الأراضي  ندرة  الموارد  والمخاطر  المختلفة  يساهم  القطاع  الزراعي  نسبته  الناتج  المحلي  الإجمالي  ويعمل  مجموع  القوى  وتشكل  الصادرات  الزراعية  مجموع  صادرات  يذهب  الأسواق  العربية  حقق  الأردن  الاكتفاء  الذاتي  عدد  المنتجات  الزراعية  كزيت  الكثير  المنتجات  الغذائية  الأساسية  كأنواع  القمح  ومشتقات  الحليب  والسكر  اللحوم  الحمراء  والخضراوات  تستورد  الخارج  نمت  الصادرات  المنتجات  ا

In [ ]:
question = "ما هي أهمية التعليم؟"

print("QUESTION:")
print(question)

print("\nQWEN ANSWER:")
print(generate_gpt_answer(question))

QUESTION:
ما هي أهمية التعليم؟

QWEN ANSWER:
أهمية  التعليم  تكمن  تطوير  المهارات  وتعزيز  الثقة  بالنفس


In [ ]:
gpt_bleu_score, gpt_predictions_df = calculate_bleu_score(
    generate_function=generate_gpt_answer,
    X_test=X_test,
    y_test=y_test,
    model_name="AraGPT2-base",
    sample_size=100
)

print("AraGPT2 BLEU Score:", gpt_bleu_score)

gpt_predictions_df.to_csv("gpt_predictions.csv", index=False)

gpt_predictions_df.head()

AraGPT2 BLEU Score: 0.09806571565847116


,Model,Question,Actual_Answer,Predicted_Answer
0,AraGPT2-base,علام تعتمد تحقيق,أعتمد تحقيق أهدافي التخطيط الجيد والعمل ...,أعتمد تحقيق الأهداف المهنية خلال ممارسة ...
1,AraGPT2-base,أهم الدول المستوردة للمنتجات الزراعية,يعتبر القطاع الزراعي القطاعات المهمة وتلع...,أهم الدول المستوردة للمنتجات الزراعية الم...
2,AraGPT2-base,أهمية التعليم,أهمية التعليم العالي تشمل تطوير المهارات ...,أهمية التعليم تكمن توفير فرص تعليمية تعز...
3,AraGPT2-base,لماذا الأكسجين,الأكسجين مهم لأنه ضروري لعملية التنفس ال...,الأكسجين ضروري لصحة الإنسان لأنه يساعد ت...
4,AraGPT2-base,مادة تعتبر موصل جيد,النحاس يعتبر موصل جيد للكهرباء,مادة تعتبر موصل جيد لأنها تحتوي مواد طب...
